In [1]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

# Udahub Application

## Core Database

**Init DB**

In [2]:
udahub_db = "data/core/udahub.db"

In [3]:
reset_db(udahub_db)

2026-06-25 10:14:31,096 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-25 10:14:31,096 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/core/udahub.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

**Account**

In [5]:
account_id = "cultpass"
account_name = "CultPass Card"

In [6]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

**Knowledge Base**

In [7]:
# Create 10 additional knowledge-base articles in memory (no file mutation)
additional_articles = [
    {
        "title": "Can I Transfer My Reservation to Someone Else?",
        "content": "Reservations are linked to the account holder and are generally non-transferable.\n\n- Ask the user to cancel the booking if they cannot attend\n- The other person should reserve through their own account\n- For accessibility or exceptional cases, escalate to human support\n\nSuggested phrasing:\n\"Reservations are usually tied to your account and cannot be transferred. If you cannot attend, please cancel your booking so another user can reserve the spot.\"",
        "tags": "reservation, transfer, policy, cancellation"
    },
    {
        "title": "What Happens If I Miss an Experience?",
        "content": "If a user misses a reserved experience:\n\n- Mark it as a no-show according to policy\n- Explain that repeated no-shows may affect booking privileges\n- Encourage cancellation in advance when possible\n\nSuggested phrasing:\n\"If you miss a reserved event without canceling, it may be treated as a no-show. Repeated no-shows can impact future reservations.\"",
        "tags": "no-show, reservation, policy, attendance"
    },
    {
        "title": "How to View Upcoming Reservations",
        "content": "Users can view all upcoming reservations in the app.\n\n- Open CultPass app\n- Go to My Bookings or Reservations\n- Select any item to view date, venue, and QR details\n\nSuggested phrasing:\n\"You can find all upcoming reservations in the My Bookings section of the CultPass app.\"",
        "tags": "reservations, app, bookings, navigation"
    },
    {
        "title": "How to Update Email or Profile Details",
        "content": "Users can update profile details from account settings.\n\n- Go to My Account > Profile\n- Edit email, name, or contact fields\n- Save changes and verify email if prompted\n\nSuggested phrasing:\n\"To update your profile, open My Account in the app, edit your details, and save. You may be asked to verify your new email.\"",
        "tags": "profile, email, account, settings"
    },
    {
        "title": "Are Premium Events Included in the Plan?",
        "content": "Premium experiences may require an additional charge.\n\n- Show users that premium pricing is displayed before confirmation\n- Standard monthly quota applies to eligible events\n- Suggest filtering by included experiences to avoid extra fees\n\nSuggested phrasing:\n\"Some premium experiences include an extra fee, which you will see before confirming your reservation.\"",
        "tags": "premium, pricing, subscription, events"
    },
    {
        "title": "Why Is an Experience Sold Out?",
        "content": "An experience may be sold out when all available slots are reserved.\n\n- Recommend joining waitlist if available\n- Suggest similar experiences in the same category\n- Ask users to check back for cancellations\n\nSuggested phrasing:\n\"This experience is currently sold out. You can join the waitlist if available or browse similar events.\"",
        "tags": "sold out, availability, waitlist, experiences"
    },
    {
        "title": "How to Contact Human Support",
        "content": "Escalate to human support when account access, billing disputes, or policy exceptions are involved.\n\n- Collect concise issue summary\n- Attach relevant reservation or transaction IDs\n- Route via in-app help or support email workflow\n\nSuggested phrasing:\n\"I can connect you with a support specialist for this issue. Please share any related booking or payment reference IDs.\"",
        "tags": "support, escalation, help, account"
    },
    {
        "title": "Can I Change the Date of My Reservation?",
        "content": "Date changes depend on event policy and slot availability.\n\n- Check if modification is enabled in booking details\n- If not, cancel and rebook another date\n- Warn users about any cancellation window rules\n\nSuggested phrasing:\n\"If date change is available, you can modify from the booking details page. Otherwise, cancel and rebook on another date.\"",
        "tags": "reservation, modify, date change, policy"
    },
    {
        "title": "How Refunds Work for Paid Add-ons",
        "content": "Refund eligibility for paid add-ons depends on cancellation timing and provider terms.\n\n- Verify event-specific refund terms\n- If outside self-service window, escalate for manual review\n- Never promise refunds before validation\n\nSuggested phrasing:\n\"Refunds for paid add-ons depend on the event policy and cancellation timing. I can help check your booking terms.\"",
        "tags": "refund, billing, add-on, policy"
    },
    {
        "title": "How to Redeem Promo Codes",
        "content": "Promo codes are applied during checkout when applicable.\n\n- Enter code in promo field before payment confirmation\n- Confirm discount is reflected in final amount\n- If invalid, verify expiry date and usage conditions\n\nSuggested phrasing:\n\"You can apply your promo code at checkout. Make sure the discount appears before you confirm payment.\"",
        "tags": "promo code, discount, checkout, billing"
    },
]

print(f"Prepared {len(additional_articles)} new in-memory articles.")

Prepared 10 new in-memory articles.


In [8]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

cultpass_articles.extend(additional_articles)

In [9]:
cultpass_articles

[{'title': 'How to Reserve a Spot for an Event',
  'content': 'If a user asks how to reserve an event:\n\n- Guide them to the CultPass app\n- Instruct them to browse the experience catalog and tap \'Reserve\'\n- If it\'s a premium or limited event, check if reservation confirmation is required via email\n- Remind them to arrive at least 15 minutes early with their QR code visible\n\n**Suggested phrasing:**\n"You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping \'Reserve\'. Be sure to arrive 15 minutes early with your QR code ready."',
  'tags': 'reservation, events, booking, attendance'},
 {'title': "What's Included in a CultPass Subscription",
  'content': 'Each user is entitled to 4 cultural experiences per month, which may include:\n- Art exhibitions\n- Museum entries\n- Music concerts\n- Film screenings and more\n\nSome premium experiences may require an additional fee (visible in the app).\n\n**Suggested phrasing:**\n"Your CultPass s

In [10]:
if len(cultpass_articles) < 14:
    raise AssertionError("You should load the articles with at least 14 records")

In [11]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb) 
    

**Ticket**

In [12]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [13]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [14]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )
    
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])


# Tests

In [15]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(account)

<Account(account_id='cultpass', account_name='CultPass Card')>


In [16]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    for article in account.knowledge_articles:
        print(article)

<Knowledge(article_id='45b53b28-fed0-438c-b082-a2fa01076840', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='9b571f14-ef60-4b73-bea3-ac86459b5d4a', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='2d3cfa57-a42d-43f0-8c4f-8e91117cdae8', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='505cc277-376f-4336-ad24-cad5cb769c18', title='How to Handle Login Issues?')>
<Knowledge(article_id='10e64e4c-0cee-4e14-8318-fd610ae60d4a', title='Can I Transfer My Reservation to Someone Else?')>
<Knowledge(article_id='a584b7c2-fe44-46f0-afab-b20e7265cd92', title='What Happens If I Miss an Experience?')>
<Knowledge(article_id='3fb43614-0370-4a22-9582-59e5e5bb4520', title='How to View Upcoming Reservations')>
<Knowledge(article_id='0dad7559-9d35-456a-bd0f-a12b8e373515', title='How to Update Email or Profile Details')>
<Knowledge(article_id='5bebb43b-ad91-4139-9eda-6734639f5485', title='Are Premium Events Included in the Plan?')>
<Knowledg

In [17]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='eae34e4e-31c2-4b98-ab78-6d122e0ab334', user_name='Alice Kingsley', external_user_id='a4ab87')>


In [18]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    
    ticket:udahub.Ticket = user.tickets[0]
    for message in ticket.messages:
        print(message)

<TicketMessage(message_id='a9ab4590-59e2-4003-b31e-471c16a5a6be', role='user', content='I can't log in to my Cultpass ...')>


# ChromaDB — Vector Store Setup

Reads all knowledge articles from  and ingests them into ChromaDB.

Run this section **after** the Core Database section above.
Re-running it will reset and rebuild the vector store from scratch.

In [19]:
import shutil
import os
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [20]:
CHROMA_PATH = "data/core/chroma/knowledge"
COLLECTION_NAME = "knowledge_articles"
EMBEDDING_MODEL = "text-embedding-3-small"

**Reset ChromaDB**

In [21]:
# Wipe and recreate the ChromaDB directory for a clean build
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)
    print(f"🗑️  Cleared existing ChromaDB at {CHROMA_PATH}")

os.makedirs(CHROMA_PATH, exist_ok=True)
print(f"✅ ChromaDB directory ready at {CHROMA_PATH}")

🗑️  Cleared existing ChromaDB at data/core/chroma/knowledge
✅ ChromaDB directory ready at data/core/chroma/knowledge


**Load articles from udahub.db**

In [22]:
# Read all knowledge articles from udahub.db
with get_session(engine) as session:
    articles = session.query(udahub.Knowledge).filter_by(
        account_id=account_id
    ).all()
    # Detach from session before converting
    articles_data = [
        {
            "article_id": a.article_id,
            "title":      a.title,
            "content":    a.content,
            "tags":       a.tags,
            "account_id": a.account_id,
        }
        for a in articles
    ]

print(f"📚 Loaded {len(articles_data)} articles from udahub.db")

📚 Loaded 14 articles from udahub.db


**Convert to LangChain Documents**

In [23]:
# Each article becomes a Document.
# page_content = title + content (for richer semantic matching)
# metadata = fields used for filtering and display in kb_tools.py
documents = [
    Document(
        page_content=f"{a['title']}\n\n{a['content']}",
        metadata={
            "article_id": a["article_id"],
            "title":      a["title"],
            "tags":       a["tags"] or "",
            "account_id": a["account_id"],
        },
    )
    for a in articles_data
]

print(f"📄 Built {len(documents)} LangChain Documents")
print("Sample document:")
print(f"  Title:   {documents[0].metadata['title']}")
print(f"  Tags:    {documents[0].metadata['tags']}")
print(f"  Content: {documents[0].page_content[:100]}...")

📄 Built 14 LangChain Documents
Sample document:
  Title:   How to Reserve a Spot for an Event
  Tags:    reservation, events, booking, attendance
  Content: How to Reserve a Spot for an Event

If a user asks how to reserve an event:

- Guide them to the Cul...


In [24]:
from dotenv import load_dotenv

In [25]:
load_dotenv()

True

**Embed and persist to ChromaDB**

In [26]:
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=os.getenv("VOCAREUM_KEY"),
    base_url="https://openai.vocareum.com/v1"
)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_PATH,
)

print(f"✅ Embedded and persisted {len(documents)} articles to ChromaDB")
print(f"   Collection : {COLLECTION_NAME}")
print(f"   Path       : {CHROMA_PATH}")
print(f"   Model      : {EMBEDDING_MODEL}")

✅ Embedded and persisted 14 articles to ChromaDB
   Collection : knowledge_articles
   Path       : data/core/chroma/knowledge
   Model      : text-embedding-3-small


**Test — semantic search**

In [27]:
# Reload from disk and run a test query to confirm ingestion worked
vectorstore_check = Chroma(
    persist_directory=CHROMA_PATH,
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME,
)

test_queries = [
    "I cannot log in to my account",
    "How do I cancel my reservation?",
    "I want a refund for my booking",
]

for query in test_queries:
    results = vectorstore_check.similarity_search_with_relevance_scores(
        query=query,
        k=2,
        filter={"account_id": account_id},
    )
    print(f"\n🔍 Query: '{query}'")
    for doc, score in results:
        print(f"   [{score:.3f}] {doc.metadata['title']}")


🔍 Query: 'I cannot log in to my account'
   [0.242] How to Handle Login Issues?
   [0.030] How to Update Email or Profile Details

🔍 Query: 'How do I cancel my reservation?'
   [0.373] Can I Change the Date of My Reservation?
   [0.347] Can I Transfer My Reservation to Someone Else?

🔍 Query: 'I want a refund for my booking'
   [0.270] How Refunds Work for Paid Add-ons
   [0.138] Can I Change the Date of My Reservation?


**Verify total document count**

In [38]:
total = vectorstore_check._collection.count()
print(f"✅ ChromaDB collection contains {total} documents")
assert total == len(documents), f"Expected {len(documents)}, got {total}"
print("✅ Document count matches — ChromaDB is ready")

✅ ChromaDB collection contains 14 documents
✅ Document count matches — ChromaDB is ready
